In [1]:
%run geo_zones

StatementMeta(, 3a7152d9-3d23-4cd7-80f6-13c0f2c172c5, 19, Finished, Available, Finished, True)

In [2]:
%run transformation

StatementMeta(, 3a7152d9-3d23-4cd7-80f6-13c0f2c172c5, 33, Finished, Available, Finished, True)

In [3]:
%run manage_tables

StatementMeta(, 3a7152d9-3d23-4cd7-80f6-13c0f2c172c5, 44, Finished, Available, Finished, True)

In [4]:
from pyspark.sql.functions import *

StatementMeta(, 3a7152d9-3d23-4cd7-80f6-13c0f2c172c5, 45, Finished, Available, Finished, False)

In [5]:
TABLE_BASE_NAME = "OpenAQ_measurements"
OPENAQ_TABLE_TYPES = ["hours", "days"]

OPENAQ_SCHEMA = {
    "station_id": {"type": "integer", "nullable": False, "primary_key": True},
    "sensor_id": {"type": "integer", "nullable": False, "primary_key": True},
    "parameter_id": {"type": "integer", "nullable": False, "primary_key": True},
    "parameter_name": {"type": "string", "nullable": False, "not_empty": True},
    "parameter_units": {"type": "string", "nullable": True},
    "parameter_display_name": {"type": "string", "nullable": True},
    "latitude": {"type": "double", "nullable": False},
    "longitude": {"type": "double", "nullable": False},
    "measurement_value": {"type": "double", "nullable": False},
    "has_flags": {"type": "boolean", "nullable": False},
    "measurement_datetime": {"type": "timestamp", "nullable": False, "primary_key": True}
}
OPENAQ_COLUMN_MAPPING = {
    "id": "station_id",
    "sensors_id": "sensor_id",
    "sensors_parameter_id": "parameter_id",
    "sensors_parameter_name": "parameter_name",
    "sensors_parameter_units": "parameter_units",
    "sensors_parameter_displayName": "parameter_display_name",
    "coordinates_latitude": "latitude",
    "coordinates_longitude": "longitude",
    "measurement_flagInfo_hasFlags": "has_flags",
    "measurement_period_datetimeFrom_local": "measurement_datetime"
}
RESULT_COLUMNS = [
    "station_id",
    "sensor_id",
    "parameter_id",
    "parameter_name",
    "parameter_units",
    "parameter_display_name",
    "measurement_value",
    "measurement_datetime",
    "location_id"
]

VALUE_RANGES = {
    "pm10": (0, None),
    "pm25": (0, None),
    "pm1": (0, None),
    "no2": (0, None),
    "co": (0, None),
    "so2": (0, None),
    "o3": (0, None),
    "no": (0, None),
    "nox": (0, None),
    "relativehumidity": (0, 100)
}

StatementMeta(, 3a7152d9-3d23-4cd7-80f6-13c0f2c172c5, 46, Finished, Available, Finished, False)

In [6]:
def remove_timezone(df, column):
    return df.withColumn(column, col(column).cast("string").substr(1, 19))

def filter_parameter_ranges(df, parameter_column, value_column, ranges):
    for parameter, (min_value, max_value) in ranges.items():
        is_param = col(parameter_column) == parameter
        is_invalid = lit(False)
        if min_value is not None:
            is_invalid = is_invalid | (col(value_column) < min_value)
        if max_value is not None:
            is_invalid = is_invalid | (col(value_column) > max_value)
        df = df.filter(~(is_param & is_invalid))
    return df

def normalize_openaq(df, column_mapping, schema):
    df = normalize_column_names(df)
    df = rename_columns(df, column_mapping)
    df = select_columns(df, schema)
    df = remove_timezone(df, "measurement_datetime")
    df = cast_columns(df, {column: prop["type"] for column, prop in schema.items()})
    return df

def filter_openaq(df, schema):
    df = drop_null_rows(df, [column for column, prop in schema.items() if not prop["nullable"]])
    df = drop_empty_rows(df, [column for column, prop in schema.items() if prop.get("not_empty", False)])
    df = df.dropDuplicates([column for column, prop in schema.items() if prop.get("primary_key", False)])
    df = filter_rows(df, "has_flags", False, "=")
    df = filter_parameter_ranges(df, "parameter_name", "measurement_value", VALUE_RANGES)
    return df
    
def finalize_locations(df, result_columns):
    df = select_columns(df, result_columns)
    df = df.filter(col("location_id").isNotNull())
    return df

StatementMeta(, 3a7152d9-3d23-4cd7-80f6-13c0f2c172c5, 47, Finished, Available, Finished, False)

In [7]:
def read_openaq(table_type):
    table_name = f"{TABLE_BASE_NAME}_{table_type}"
    return read_table(table_name, "Bronze_lakehouse")

def load_and_process_openaq_table(table_type, column_mapping, schema):
    df = read_openaq(table_type)
    df = normalize_openaq(df, column_mapping, schema)
    df = filter_openaq(df, schema)
    return df

StatementMeta(, 3a7152d9-3d23-4cd7-80f6-13c0f2c172c5, 48, Finished, Available, Finished, False)

In [8]:
def process(table_type):
    sedona = prepare_spatial_processing(spark, TAXI_ZONES_SHAPEFILE_FOLDER_PATH)
    df = load_and_process_openaq_table(table_type, OPENAQ_COLUMN_MAPPING, OPENAQ_SCHEMA)
    df = add_zone_from_coordinates(df, sedona, "latitude", "longitude")
    df = finalize_locations(df, RESULT_COLUMNS)
    merge_table(df, f"{TABLE_BASE_NAME}_{table_type}", [column for column, prop in OPENAQ_SCHEMA.items() if prop.get("primary_key", False)])
    return df

StatementMeta(, 3a7152d9-3d23-4cd7-80f6-13c0f2c172c5, 49, Finished, Available, Finished, False)

In [9]:
for table_type in OPENAQ_TABLE_TYPES:
    process(table_type)

StatementMeta(, 3a7152d9-3d23-4cd7-80f6-13c0f2c172c5, 50, Finished, Available, Finished, False)